# 🟣 YOLOv8 Segmentation Training
Ringkasan notebook untuk melatih model segmentasi (task=segment). Ikuti urutan sel dari atas ke bawah.



In [ ]:
import os
import torch
from ultralytics import YOLO

# Cek lingkungan
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch version: {torch.__version__}")
print(f"Ultralytics version: {YOLO.__version__}")



In [ ]:
# --- Konfigurasi dataset & model ---
# TODO: sesuaikan path di bawah dengan lokasi dataset di mesin ini (punya GPU)
data_yaml = r"G:/path/to/your/segmentation/data.yaml"  # contoh: G:/datasets/myseg/data.yaml
base_model = "yolov8n-seg.pt"  # bisa diganti dengan checkpoint lain, misal runs/segment/.../best.pt
project_dir = "runs/segment"
run_name = "yolov8n-seg-custom"

print("Data YAML exists:", os.path.exists(data_yaml))
print("Base model exists:", os.path.exists(base_model))
if os.path.exists(data_yaml):
    with open(data_yaml, "r") as f:
        print("\nIsi data.yaml:\n", f.read())



In [ ]:
# --- Hyperparameter & training config ---
training_config = {
    "data": data_yaml,
    "model": base_model,
    "epochs": 50,
    "imgsz": 640,
    "batch": 8,
    "name": run_name,
    "project": project_dir,
    "exist_ok": True,
    # Optimizer / scheduler
    "lr0": 0.005,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "patience": 20,
    # Augmentations (sesuaikan jika perlu)
    "hsv_h": 0.02,
    "hsv_s": 0.7,
    "hsv_v": 0.5,
    "degrees": 5.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 2.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.05,
    # Saving
    "save": True,
    "save_period": 10,
    "cache": False,
    "workers": 4,
}

print("Training config:")
for k, v in training_config.items():
    print(f"  {k}: {v}")



In [ ]:
# --- Jalankan training (task=segment) ---
model = YOLO(training_config["model"])

print(f"Mulai training: {training_config['name']}")
results = model.train(
    task="segment",
    data=training_config["data"],
    epochs=training_config["epochs"],
    imgsz=training_config["imgsz"],
    batch=training_config["batch"],
    name=training_config["name"],
    project=training_config["project"],
    exist_ok=training_config["exist_ok"],
    lr0=training_config["lr0"],
    lrf=training_config["lrf"],
    momentum=training_config["momentum"],
    weight_decay=training_config["weight_decay"],
    warmup_epochs=training_config["warmup_epochs"],
    warmup_momentum=training_config["warmup_momentum"],
    warmup_bias_lr=training_config["warmup_bias_lr"],
    patience=training_config["patience"],
    hsv_h=training_config["hsv_h"],
    hsv_s=training_config["hsv_s"],
    hsv_v=training_config["hsv_v"],
    degrees=training_config["degrees"],
    translate=training_config["translate"],
    scale=training_config["scale"],
    shear=training_config["shear"],
    perspective=training_config["perspective"],
    flipud=training_config["flipud"],
    fliplr=training_config["fliplr"],
    mosaic=training_config["mosaic"],
    mixup=training_config["mixup"],
    save=training_config["save"],
    save_period=training_config["save_period"],
    cache=training_config["cache"],
    workers=training_config["workers"],
)

print("\n✅ Training selesai")
print(f"Model tersimpan di: runs/segment/{training_config['name']}/weights/best.pt")



In [ ]:
# --- Validasi model ---
trained_model_path = f"runs/segment/{training_config['name']}/weights/best.pt"
val_model = YOLO(trained_model_path)

print(f"Load model terlatih: {trained_model_path}")
print("Model exists:", os.path.exists(trained_model_path))

val_results = val_model.val(
    task="segment",
    data=training_config["data"],
    imgsz=training_config["imgsz"],
    batch=training_config["batch"],
    save_json=True,
    save_hybrid=True,
    plots=True,
)

print("\n📈 Hasil validasi:")
print(f"mAP@0.5: {val_results.seg.map50:.4f}")
print(f"mAP@0.5-0.95: {val_results.seg.map:.4f}")
print(f"Precision: {val_results.seg.mp:.4f}")
print(f"Recall: {val_results.seg.mr:.4f}")



In [ ]:
# --- Contoh inferensi/predict ---
sample_source = r"datasets/potholes_raw/valid/images"  # ganti dengan folder/gambar/video Anda
predict_run = f"{training_config['name']}-predict"

pred_results = val_model.predict(
    task="segment",
    source=sample_source,
    imgsz=training_config["imgsz"],
    conf=0.25,
    iou=0.45,
    save=True,
    save_txt=True,
    save_conf=True,
    project=project_dir,
    name=predict_run,
    exist_ok=True,
)

print(f"✅ Predict selesai. Output: {project_dir}/{predict_run}")
print(f"Jumlah hasil: {len(pred_results)}")



## Catatan pemakaian
- Pastikan `data_yaml` menunjuk ke dataset segmentasi (format YOLOv8: polygon normalisasi).
- Ganti `base_model` bila punya checkpoint sebelumnya (mis. `runs/segment/.../best.pt`).
- Jalankan sel sesuai urutan: cek environment → konfigurasi dataset → config training → train → val → predict.
- Jika ingin mengubah hyperparameter, edit `training_config` lalu jalankan ulang sel training.

